# Data Acquisition (XJTU Downsampling)
**Goal:** Aggregate and downsample the raw XJTU vibration data so that 1 CSV file = 1 Bearing (where 1 row represents 1 minute of operation).

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm

# ==========================================
# CONFIGURATION
# ==========================================
# Define input and output paths
RAW_DATA_PATH = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets"
OUTPUT_PATH = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets\Processed_Data\Downsampled"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Dataset Specifics
# XJTU: 25.6 kHz, ~1.28 sec snapshot (32,768 rows) every 1 minute.
# We will simply load and structure these files.
ROWS_PER_FILE = 32768

print(f"Raw Data Path: {os.path.abspath(RAW_DATA_PATH)}")
print(f"Output Path: {os.path.abspath(OUTPUT_PATH)}")

Raw Data Path: /home/praktikan/projects/github/DwiAnggara/ProyekRisetBearing/XJTU-SY_Bearing_Datasets
Output Path: /home/praktikan/projects/github/DwiAnggara/ProyekRisetBearing/XJTU-SY_Bearing_Datasets/Processed_Data/Downsampled


In [ ]:
def process_and_combine_bearing_data():
    """
    Loops through specific condition folders ('35Hz12kN', '37.5Hz11kN', '40Hz10kN')
    and their raw bearing files. Aggregates the raw arrays of vibration data.
    1 data file = 1 Bearing (1 row = 1 minute of operation = 1 raw CSV file).
    """
    if not os.path.exists(RAW_DATA_PATH):
        print(f"Warning: Raw data path '{RAW_DATA_PATH}' does not exist.")
        return

    # Specific Target Folders Based on Final Blueprint
    target_conditions = ['35Hz12kN', '37.5Hz11kN', '40Hz10kN']
    
    summary_info = []
    last_df = None
    
    for condition in target_conditions:
        cond_path = os.path.join(RAW_DATA_PATH, condition)
        if not os.path.exists(cond_path):
            print(f"Skipping {condition}, folder not found.")
            continue
            
        bearing_folders = [f for f in os.listdir(cond_path) if os.path.isdir(os.path.join(cond_path, f))]
        
        for bearing in tqdm(bearing_folders, desc=f"Processing {condition}"):
            bearing_path = os.path.join(cond_path, bearing)
            csv_files = sorted(glob.glob(os.path.join(bearing_path, "*.csv")))
            
            if not csv_files:
                continue
                
            bearing_data_list = []
            
            for minute_idx, file in enumerate(csv_files):
                try:
                    df_min = pd.read_csv(file)
                except Exception as e:
                    print(f"Error reading {file}: {e}")
                    continue
                
                # Extract signals based on typical XJTU structure
                if 'Horizontal_vibration_signals' in df_min.columns:
                    h_sig = df_min['Horizontal_vibration_signals'].values
                    v_sig = df_min['Vertical_vibration_signals'].values
                else:
                    h_sig = df_min.iloc[:, 0].values
                    v_sig = df_min.iloc[:, 1].values if df_min.shape[1] > 1 else np.zeros_like(h_sig)
                
                # Package 1 row per minute
                minute_record = {
                    'Minute': minute_idx + 1,
                    'H_Signal': h_sig,
                    'V_Signal': v_sig
                }
                bearing_data_list.append(minute_record)
            
            if bearing_data_list:
                final_df = pd.DataFrame(bearing_data_list)
                
                cond_out_path = os.path.join(OUTPUT_PATH, condition)
                os.makedirs(cond_out_path, exist_ok=True)
                
                output_filename = os.path.join(cond_out_path, f"{bearing}_downsampled.pkl")
                final_df.to_pickle(output_filename)
                
                file_size_mb = os.path.getsize(output_filename) / (1024 * 1024)
                summary_info.append({
                    "Condition": condition, 
                    "Bearing": bearing, 
                    "Rows": len(final_df), 
                    "Size (MB)": round(file_size_mb, 2)
                })
                last_df = final_df
                print(f"Saved: {output_filename} (Rows: {len(final_df)})")

    print("\n--- Summary ---")
    if summary_info:
        summary_df = pd.DataFrame(summary_info)
        print(summary_df.to_string(index=False))
    else:
        print("No data processed.")
        
    if last_df is not None:
        print(f"\n--- Sample Data (First 5 Rows) ---")
        print(last_df.head(5))

# Execute the pipeline
if __name__ == "__main__":
    process_and_combine_bearing_data()

Processing Condition_1:  20%|██        | 1/5 [00:07<00:30,  7.56s/it]

Saved: /home/praktikan/projects/github/DwiAnggara/ProyekRisetBearing/XJTU-SY_Bearing_Datasets/Processed_Data/Downsampled/Condition_1/Bearing1_2_downsampled.csv (Rows: 161)


Processing Condition_1:  40%|████      | 2/5 [00:14<00:22,  7.44s/it]

Saved: /home/praktikan/projects/github/DwiAnggara/ProyekRisetBearing/XJTU-SY_Bearing_Datasets/Processed_Data/Downsampled/Condition_1/Bearing1_3_downsampled.csv (Rows: 158)


Processing Condition_1:  60%|██████    | 3/5 [00:20<00:13,  6.63s/it]

...

Saved: /home/praktikan/projects/github/DwiAnggara/ProyekRisetBearing/XJTU-SY_Bearing_Datasets/Processed_Data/Downsampled/Condition_2/Bearing2_1_downsampled.csv (Rows: 491)


Processing Condition_2:  80%|████████  | 4/5 [01:05<00:19, 19.33s/it]

Saved: /home/praktikan/projects/github/DwiAnggara/ProyekRisetBearing/XJTU-SY_Bearing_Datasets/Processed_Data/Downsampled/Condition_2/Bearing2_3_downsampled.csv (Rows: 533)


Processing Condition_2: 100%|██████████| 5/5 [01:12<00:00, 14.49s/it]

Saved: /home/praktikan/projects/github/DwiAnggara/ProyekRisetBearing/XJTU-SY_Bearing_Datasets/Processed_Data/Downsampled/Condition_2/Bearing2_2_downsampled.csv (Rows: 161)

--- Summary ---
  Condition    Bearing  Rows  Size (MB)
Condition_1 Bearing1_2   161     192.05
Condition_1 Bearing1_3   158     190.34
Condition_1 Bearing1_1   123     147.87
Condition_1 Bearing1_4   122     147.69
Condition_1 Bearing1_5    52      62.34
Condition_2 Bearing2_4    42      50.44
Condition_2 Bearing2_5   339     404.26
Condition_2 Bearing2_1   491     593.89
Condition_2 Bearing2_3   533     639.47
Condition_2 Bearing2_2   161     192.37

--- Sample Data (First 5 Rows) ---
   Minute                                           H_Signal  \
0       1  0.5141377076506615,0.4624009132385254,-0.66266...   
1       2  -0.6022096052765846,0.4279493913054466,-0.6423...   
2       3  -0.4599928855895996,-0.1279473025351762,0.4808...   
3       4  1.1505365371704102,1.6330242156982422,3.386723...   
4       5  0.81